In [16]:
import pandas as pd

df_categorias = pd.read_parquet("categorias.parquet")
df_libros = pd.read_parquet("libros.parquet")

df_categorias.head()

,categoria,url_categoria,cantidad_libros,fecha_extraccion,extraido_por
0,Travel,https://books.toscrape.com/catalogue/category/...,11,2026-08-15T13:49:07,Pablo y Pedro
1,Mystery,https://books.toscrape.com/catalogue/category/...,32,2026-08-15T13:49:22,Pablo y Pedro
2,Historical Fiction,https://books.toscrape.com/catalogue/category/...,26,2026-08-15T13:49:36,Pablo y Pedro
3,Sequential Art,https://books.toscrape.com/catalogue/category/...,75,2026-08-15T13:50:17,Pablo y Pedro
4,Classics,https://books.toscrape.com/catalogue/category/...,19,2026-08-15T13:50:29,Pablo y Pedro


In [17]:
df_libros.head()

,upc,titulo,categoria,descripcion,tipo_producto,precio_sin_impuesto,precio_con_impuesto,impuesto,moneda,disponibilidad,cantidad_stock,calificacion,cantidad_resenas,url_libro,url_imagen,fecha_extraccion,extraido_por
0,a22124811bfa8350,It's Only the Himalayas,Travel,"“Wherever you go, whatever you do, just . . . ...",Books,45.17,45.17,0.0,GBP,In stock (19 available),19,2,0,https://books.toscrape.com/catalogue/its-only-...,https://books.toscrape.com/media/cache/6d/41/6...,2026-08-15T13:49:03,Pablo y Pedro
1,ce60436f52c5ee68,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,Acclaimed travel writer Rick Antonson sets his...,Books,49.43,49.43,0.0,GBP,In stock (15 available),15,4,0,https://books.toscrape.com/catalogue/full-moon...,https://books.toscrape.com/media/cache/fe/8a/f...,2026-08-15T13:49:04,Pablo y Pedro
2,f9705c362f070608,See America: A Celebration of Our National Par...,Travel,To coincide with the 2016 centennial anniversa...,Books,48.87,48.87,0.0,GBP,In stock (14 available),14,3,0,https://books.toscrape.com/catalogue/see-ameri...,https://books.toscrape.com/media/cache/c7/1a/c...,2026-08-15T13:49:04,Pablo y Pedro
3,1809259a5a5f1d8d,Vagabonding: An Uncommon Guide to the Art of L...,Travel,With a new foreword by Tim Ferriss •There’s no...,Books,36.94,36.94,0.0,GBP,In stock (8 available),8,2,0,https://books.toscrape.com/catalogue/vagabondi...,https://books.toscrape.com/media/cache/ca/30/c...,2026-08-15T13:49:05,Pablo y Pedro
4,a94350ee74deaa07,Under the Tuscan Sun,Travel,A CLASSIC FROM THE BESTSELLING AUTHOR OF UNDER...,Books,37.33,37.33,0.0,GBP,In stock (7 available),7,3,0,https://books.toscrape.com/catalogue/under-the...,https://books.toscrape.com/media/cache/45/21/4...,2026-08-15T13:49:05,Pablo y Pedro


## 1. ¿Cuántas categorías de libros existen?

In [18]:
num_categorias = df_categorias["categoria"].nunique()
print(f"Número de categorías: {num_categorias}")

Número de categorías: 50


## 2. ¿Cuántos libros hay en cada categoría?

In [19]:
libros_por_categoria = df_libros.groupby("categoria").size().reset_index(name="cantidad_libros")
libros_por_categoria = libros_por_categoria.sort_values("cantidad_libros", ascending=False).reset_index(drop=True)
libros_por_categoria

,categoria,cantidad_libros
0,Default,152
1,Nonfiction,110
2,Sequential Art,75
3,Add a comment,67
4,Fiction,65
5,Young Adult,54
6,Fantasy,48
7,Romance,35
8,Mystery,32
9,Food and Drink,30


## 3. ¿Cuál es el libro más caro? 

In [20]:
precio_maximo = df_libros["precio_con_impuesto"].max()
libros_mas_caros = df_libros[df_libros["precio_con_impuesto"] == precio_maximo]
print(f"Precio máximo (con impuesto): {precio_maximo}")
libros_mas_caros[["upc", "titulo", "categoria", "precio_con_impuesto"]]

Precio máximo (con impuesto): 59.99


,upc,titulo,categoria,precio_con_impuesto
191,9cc207168a03470d,The Perfect Play (Play by Play #1),Romance,59.99


## 4. ¿Hay algún libro que aparezca en más de una categoría? 

In [21]:
conteo_upc = df_libros.groupby("upc")["categoria"].nunique()
upcs_en_varias_categorias = conteo_upc[conteo_upc > 1]

if upcs_en_varias_categorias.empty:
    print("No hay ningún libro que aparezca en más de una categoría.")
else:
    print(f"Hay {len(upcs_en_varias_categorias)} libros que aparecen en más de una categoría:")
    display(df_libros[df_libros["upc"].isin(upcs_en_varias_categorias.index)][["upc", "titulo", "categoria"]].sort_values("upc"))

No hay ningún libro que aparezca en más de una categoría.


## 5. ¿Cuál es el libro más barato de cada categoría? 

In [22]:
precio_min_por_categoria = df_libros.groupby("categoria")["precio_con_impuesto"].transform("min")
libros_mas_baratos = df_libros[df_libros["precio_con_impuesto"] == precio_min_por_categoria]
libros_mas_baratos = libros_mas_baratos.sort_values(["categoria", "titulo"]).reset_index(drop=True)
libros_mas_baratos[["categoria", "titulo", "precio_con_impuesto"]]

,categoria,titulo,precio_con_impuesto
0,Academic,Logan Kade (Fallen Crest High #5.5),13.12
1,Add a comment,The Tipping Point: How Little Things Can Make ...,10.02
2,Adult Fiction,Fifty Shades Freed (Fifty Shades #3),15.36
3,Art,History of Beauty,10.29
4,Autobiography,The Argonauts,10.93
5,Biography,Louisa: The Extraordinary Life of Mrs. Adams,16.85
6,Business,The Third Wave: An Entrepreneur’s Vision of th...,12.61
7,Childrens,Counting Thyme,10.62
8,Christian,Blue Like Jazz: Nonreligious Thoughts on Chris...,25.77
9,Christian Fiction,Counted With the Stars (Out from Egypt #1),17.97


## 6. Diferencia entre el precio de cada libro y el precio promedio de su categoría

`diferencia = precio del libro - precio promedio de su categoría`

- Positivo -> el libro es más caro que el promedio de su categoría.
- Negativo -> el libro es más barato que el promedio de su categoría.
- Cero -> cuesta lo mismo que el promedio.

In [23]:
promedio_por_categoria = df_libros.groupby("categoria")["precio_con_impuesto"].transform("mean")
df_libros["diferencia"] = df_libros["precio_con_impuesto"] - promedio_por_categoria

df_libros[["upc", "titulo", "categoria", "precio_con_impuesto", "diferencia"]].sort_values("diferencia", ascending=False).head(10)

,upc,titulo,categoria,precio_con_impuesto,diferencia
925,6478ccb4416e6a5d,The Barefoot Contessa Cookbook,Food and Drink,59.92,28.505333
964,9c4d061c1e2fe6bf,The Bone Hunters (Lexy Vaughan & Steven Macaul...,Thriller,59.71,28.276364
25,49b24c6a41b82bd2,Boar Island (Anna Pigeon #19),Mystery,59.48,27.760937
191,9cc207168a03470d,The Perfect Play (Play by Play #1),Romance,59.99,26.056286
38,b0913e67d38d5ed5,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,Mystery,57.70,25.980938
845,396385e3de5d18c3,Civilization and Its Discontents,Psychology,59.95,25.731429
368,54fc03f1e1d355db,The Diary of a Young Girl,Nonfiction,59.90,25.639818
294,37c0cb19713d8dda,The White Cat and the Monk: A Retelling of the...,Childrens,58.08,25.441724
392,60376aa71be66083,The Man Who Mistook His Wife for a Hat and Oth...,Nonfiction,59.45,25.189818
203,6e712ea24e77bd96,Listen to Me (Fusion #1),Romance,58.99,25.056286


## 7. Libro con mayor ingreso potencial dentro de cada categoría

`ingreso potencial = precio_con_impuesto × cantidad_stock`



In [24]:
df_libros["ingreso_potencial"] = df_libros["precio_con_impuesto"] * df_libros["cantidad_stock"]

ingreso_max_por_categoria = df_libros.groupby("categoria")["ingreso_potencial"].transform("max")
libros_mayor_ingreso = df_libros[df_libros["ingreso_potencial"] == ingreso_max_por_categoria]
libros_mayor_ingreso = libros_mayor_ingreso.sort_values(["categoria", "titulo"]).reset_index(drop=True)
libros_mayor_ingreso[["categoria", "titulo", "precio_con_impuesto", "cantidad_stock", "ingreso_potencial"]]

,categoria,titulo,precio_con_impuesto,cantidad_stock,ingreso_potencial
0,Academic,Logan Kade (Fallen Crest High #5.5),13.12,5,65.60
1,Add a comment,Judo: Seven Steps to Black Belt (an Introducto...,53.90,16,862.40
2,Adult Fiction,Fifty Shades Freed (Fifty Shades #3),15.36,3,46.08
3,Art,Wall and Piece,44.18,18,795.24
4,Autobiography,Lab Girl,40.85,11,449.35
5,Biography,Benjamin Franklin: An American Life,48.19,7,337.33
6,Business,The Dirty Little Secrets of Getting Your Dream...,33.34,19,633.46
7,Childrens,Birdsong: A Story in Pictures,54.64,19,1038.16
8,Christian,(Un)Qualified: How God Uses Broken People to D...,54.00,16,864.00
9,Christian Fiction,Close to You,49.46,15,741.90
